# ATML PA2 – GPU runner (Colab / Kaggle)

Thin driver only: every experiment is a Python module in the repo (no hidden notebook state).
`outputs/` (adapters) and `results/` (JSON/CSV logs) are symlinked to persistent storage so a
disconnect does not lose finished runs; every Task 1 script skips stages that already finished.

**Runtime → Change runtime type → GPU** (T4 works; L4/A100 is ~2–4× faster).

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No CUDA GPU in this kernel. In Colab: Runtime > Change runtime type > T4 GPU. "
    "In VS Code: Select Kernel > Colab > New Colab Server > GPU (needs the Google Colab extension). "
    "A Mac cannot run this assignment's training code.")
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

AssertionError: No CUDA GPU in this kernel. In Colab: Runtime > Change runtime type > T4 GPU. In VS Code: Select Kernel > Colab > New Colab Server > GPU (needs the Google Colab extension). A Mac cannot run this assignment's training code.

In [ ]:
# ---- edit these two lines ----
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # your public PA2 repo
PERSIST  = "/content/drive/MyDrive/ATML_PA2"                       # Kaggle: "/kaggle/working/persist"

import os, sys
if PERSIST.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(f"{PERSIST}/outputs", exist_ok=True)
os.makedirs(f"{PERSIST}/results", exist_ok=True)

In [ ]:
%cd /content
!rm -rf pa2 && git clone -q $REPO_URL pa2
%cd /content/pa2
!git log --oneline -3
# persistent outputs/results
!rm -rf outputs results && ln -s $PERSIST/outputs outputs && ln -s $PERSIST/results results
!ls -la | grep -E "outputs|results"

In [ ]:
# Colab already has a CUDA torch; install the pinned rest of the stack.
!pip -q install transformers==4.57.1 tokenizers==0.22.1 trl==0.27.2 peft==0.17.1 "datasets>=3.0,<5" "accelerate>=1.0,<2" "bitsandbytes>=0.45" pyyaml pandas scikit-learn matplotlib
!python -m scripts.download_assets
!python -m scripts.validate_assets
!python -m tests.test_objectives

## Task 1 – DPO
Approximate T4 times: standard train ~40–60 min, each eval ~10 min, each β fork ~20 min, length-balanced train ~35 min.
Run cells in order; re-running a cell after a disconnect resumes (finished stages are skipped).

In [ ]:
# 1a. Standard DPO: one epoch on the fixed preference set
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard

In [ ]:
# 1b. Common evaluation protocol: SFT baseline and standard DPO
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter none --name sft
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

In [ ]:
# 2. Beta study: short forks (first 600 rows) for beta in {0.03, 0.10, 0.30}; writes results/task1_dpo/beta_summary.csv
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

In [ ]:
# 3. Length-confounding study: dataset stats, length-balanced training, per-stratum evaluation
!python -m task1_dpo.analyze_length --config configs/dpo.yaml

In [ ]:
# Qualitative candidates (you pick and judge the examples for the report)
!python -m task1_dpo.qualitative --config configs/dpo.yaml
import pandas as pd
display(pd.read_csv("results/task1_dpo/beta_summary.csv"))
display(pd.read_csv("results/task1_dpo/length_strata.csv"))
display(pd.read_csv("results/task1_dpo/length_generation.csv"))

## Getting results back into the repo
`results/` lives in your Drive folder (`ATML_PA2/results`). Copy it into your local repo's `results/`
and commit it — these small JSON/CSV files are the machine-readable evidence the report must trace to.
Do **not** commit `outputs/` (adapters); it is git-ignored.